# Agente Auditor (Agent A) — demostración en vivo

**AI Governance · Segunda línea de defensa**

## El problema

*Agent B* ya está desplegado en los canales digitales: evalúa solicitudes de pólizas y
pre-aprueba el pago de siniestros menores consultando la base de conocimiento (RAG).
Decide solo, en segundos, sobre dinero de la compañía.

La pregunta de gobierno es simple: **¿quién audita a Agent B?**

*Agent A* es ese control. Revisa cada decisión contra las reglas del negocio, la seguridad
tecnológica y los límites de riesgo, **antes de que el impacto llegue a producción**.

## Qué se va a ver

Para cada una de las 4 interacciones del log, el mismo flujo de cinco etapas:

| Etapa | Pregunta que responde |
|---|---|
| 1. Decisión | ¿Qué hizo Agent B? |
| 2. Razonamiento | ¿Qué controles se dispararon y con qué evidencia? |
| 3. Métrica | ¿Qué tan fiel fue la decisión al contexto normativo? |
| 4. Confianza | ¿Qué tan seguro está el auditor de su propio veredicto? |
| 5. Similitud | ¿La respuesta se apoya en el contexto que se recuperó? |

Todo corre con la **librería estándar de Python**: sin modelos externos, sin llamadas de red y
sin costo por transacción. El puntaje se recalcula a mano y es reproducible ante un regulador.

---
## 0. Preparación del entorno

Se carga el motor de auditoría y la configuración de control. Las huellas SHA-256 de `reglas.json`
y del archivo de casos viajan en cada reporte: es lo que permite demostrar, meses después, con qué
versión de las reglas se auditó una transacción.

In [ ]:
import sys
import textwrap
from pathlib import Path


def raiz_proyecto() -> Path:
    '''Ubica la raíz del repositorio desde cualquier carpeta de trabajo.'''
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / 'auditor' / '__init__.py').exists() and (carpeta / 'reglas.json').exists():
            return carpeta
    raise FileNotFoundError('No se encontró la raíz del proyecto (auditor/ + reglas.json)')


RAIZ = raiz_proyecto()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from auditor.config import cargar_reglas
from auditor.confianza import confianza_decision, similitud_semantica
from auditor.entrada import cargar_casos
from auditor.fidelidad import crear_evaluador
from auditor.motor import Auditor
from auditor.reporte import formatear_reporte, formatear_veredicto

ANCHO = 100

REGLAS = cargar_reglas(RAIZ / 'reglas.json')
CASOS, SHA_CASOS = cargar_casos(RAIZ / 'data' / 'casos.json')
AUDITOR = Auditor(REGLAS)

activos = sum(1 for control in REGLAS.controles if control.activo)
print(f'Proyecto         : {RAIZ}')
print(f'Reglas v{REGLAS.version}    : {activos} controles activos   SHA-256 {REGLAS.sha256[:12]}...')
print(f'Casos            : {len(CASOS)} interacciones de Agent B   SHA-256 {SHA_CASOS[:12]}...')
print(f'Evaluador activo : {REGLAS.fidelidad.evaluador_por_defecto}')
print(f'Umbral de revisión: {REGLAS.fidelidad.umbral_revision:.2f}   Pesos: {REGLAS.fidelidad.pesos}')

---
## 1. El dataset de entrada

Cada registro es una interacción real de Agent B: lo que la póliza decía (**contexto RAG**) y lo
que el agente le respondió al cliente. El auditor no ve nada más; esa es exactamente la evidencia
con la que trabajaría en producción.

In [ ]:
def bloque(titulo: str, texto: str, sangria: str = '      ') -> None:
    print(f'   {titulo}')
    for linea in textwrap.wrap(texto, width=ANCHO - len(sangria)):
        print(f'{sangria}{linea}')


for caso in CASOS:
    print('=' * ANCHO)
    print(f'CASO {caso.id_caso}')
    print('-' * ANCHO)
    bloque('CONTEXTO RAG   (lo que la póliza dice)', caso.contexto_rag)
    bloque('RESPUESTA B    (lo que el agente hizo)', caso.respuesta_agent_b)
    print()

---
## 2. El flujo de auditoría

Las cinco etapas que se ejecutan por cada caso:

1. **Decisión** — se clasifica la acción de Agent B: aprobar, rechazar, escalar a un humano o
   *desconocida*. Las negaciones tienen prioridad, para que "no cubrimos" nunca se lea como una
   aprobación.
2. **Razonamiento** — se ejecutan los controles deterministas (topes, deducible, límite por edad,
   fraude, listas restrictivas, alucinación). Cada uno deja su evidencia numérica.
3. **Métrica** — el Índice de Fidelidad Analítica, descompuesto en sus cuatro componentes.
4. **Confianza** — qué tan sólida es la evidencia sobre la que el propio auditor se pronuncia.
   Confianza baja no significa que Agent B falló: significa que este control no alcanza y debe
   mirarlo un humano.
5. **Similitud** — anclaje temático entre contexto y respuesta, como señal complementaria.

La celda siguiente define la capa de presentación. No contiene lógica de negocio: toda la
decisión vive en el paquete `auditor`, ya probado con 99 pruebas automáticas.

In [ ]:
MARCAS = {
    'CUMPLE': '[OK]', 'NO_CUMPLE': '[!!]', 'INDETERMINADO': '[??]', 'NO_APLICA': '[--]',
    'SOPORTADA': '[OK]', 'NO_SOPORTADA': '[!!]', 'NO_VERIFICABLE': '[--]',
}


def etapa(numero: int, nombre: str) -> None:
    print()
    print(f'--- {numero}/5  {nombre} '.ljust(ANCHO, '-'))


def _num(valor, formato: str = '{:.3f}', vacio: str = 'no definido') -> str:
    return vacio if valor is None else formato.format(valor)


def mostrar_decision(veredicto) -> None:
    etapa(1, 'DECISIÓN DE AGENT B')
    claves = ', '.join(veredicto.accion.evidencia) or 'ninguna'
    print(f'   Acción clasificada : {veredicto.accion.tipo}')
    print(f'   Frases detectadas  : {claves}')


def mostrar_razonamiento(veredicto) -> None:
    etapa(2, 'RAZONAMIENTO DEL AUDITOR (controles deterministas)')
    encabezado = f'   {"CONTROL":<10}{"ESTADO":<22}{"SEVERIDAD":<20}TIPO DE CONTROL'
    print(encabezado)
    print('   ' + '-' * (ANCHO - 6))
    for resultado in veredicto.resultados:
        estado = f'{MARCAS[resultado.estado.value]} {resultado.estado.value}'
        print(f'   {resultado.id:<10}{estado:<22}{resultado.severidad:<20}{resultado.tipo}')
        for linea in textwrap.wrap(resultado.mensaje, width=ANCHO - 10):
            print(f'         {linea}')

In [ ]:
def mostrar_metrica(veredicto) -> None:
    etapa(3, 'MÉTRICA: Índice de Fidelidad Analítica')
    fidelidad = veredicto.fidelidad
    componentes, pesos = fidelidad['componentes'], fidelidad['pesos_efectivos']
    c = componentes['C'] or 0.0
    a = componentes['A']
    t = componentes['T'] or 0.0
    p = componentes['P'] or 0.0

    print('   Componentes')
    print(f'      C  Cumplimiento normativo : {_num(c)}     peso {pesos["cumplimiento"]:.2f}')
    print(f'      A  Anclaje factual        : {_num(a)}     peso {pesos["anclaje"]:.2f}')
    print(f'      T  Cobertura temática     : {_num(t)}     peso {pesos["tematico"]:.2f}')
    print(f'      P  Penalización           : {_num(p)}     (multiplicativa)')
    if a is None:
        print('      [!!] Sin aserciones verificables independientes: los pesos se renormalizan')
        print(f'           y el caso queda con cobertura_evidencia = {fidelidad["cobertura_evidencia"]}')

    print()
    print('   Aserciones contrastadas contra el contexto')
    if not fidelidad['aserciones']:
        print('      (ninguna: la respuesta no afirma nada contrastable)')
    for asercion in fidelidad['aserciones']:
        marca = MARCAS[asercion['estado']]
        etiqueta = f'      {marca} {asercion["familia"]:<13}{asercion["texto"]:<30}'
        motivo = textwrap.wrap(asercion['motivo'], width=max(30, ANCHO - len(etiqueta)))
        print(f'{etiqueta}{motivo[0] if motivo else ""}')
        for continuacion in motivo[1:]:
            print(f'{" " * len(etiqueta)}{continuacion}')

    if fidelidad['penalizaciones']:
        print()
        print('   Penalizaciones activadas')
        for nombre, peso in fidelidad['penalizaciones'].items():
            print(f'      [!!] {nombre:<30} -{peso:.2f}')

    a_calculo = 0.0 if a is None else a
    base = pesos['cumplimiento'] * c + pesos['anclaje'] * a_calculo + pesos['tematico'] * t
    calculado = round(base * (1 - p), 2)
    print()
    print('   Cálculo (reproducible a mano)')
    print(f'      IFA = ({pesos["cumplimiento"]:.2f} x {c:.3f} + {pesos["anclaje"]:.2f} x '
          f'{a_calculo:.3f} + {pesos["tematico"]:.2f} x {t:.3f}) x (1 - {p:.2f})')
    print(f'          = {base:.4f} x {1 - p:.2f} = {base * (1 - p):.4f}   ->   {calculado:.2f}')
    coincide = '[OK]' if abs(calculado - veredicto.indice) < 0.01 else '[!!]'
    print(f'      {coincide} índice reportado por el motor: {veredicto.indice:.2f}')


def mostrar_confianza(veredicto) -> None:
    etapa(4, 'CONFIANZA DEL AUDITOR EN SU PROPIO VEREDICTO')
    confianza = confianza_decision(veredicto, REGLAS)
    print(f'   Confianza : {confianza["valor"]:.2f}   [banda {confianza["banda"]}]')
    print('   Señales que la sustentan')
    for nombre, aporte in confianza['factores'].items():
        print(f'      - {nombre:<28}{aporte:+.2f}')
    print()
    for linea in textwrap.wrap(confianza['explicacion'], width=ANCHO - 6):
        print(f'      {linea}')


def mostrar_similitud(caso, veredicto) -> None:
    etapa(5, 'SIMILITUD SEMÁNTICA  CONTEXTO <-> RESPUESTA')
    similitud = similitud_semantica(caso.contexto_rag, caso.respuesta_agent_b, REGLAS)
    print(f'   Similitud : {similitud["valor"]:.3f}   (backend: {similitud["backend"]})')
    comunes = ', '.join(similitud['tokens_comunes']) or 'ninguno'
    print(f'   Términos compartidos : {comunes}')
    print()
    for linea in textwrap.wrap(similitud['detalle'], width=ANCHO - 6):
        print(f'      {linea}')


def mostrar_veredicto(veredicto) -> None:
    print()
    print('=' * ANCHO)
    print('VEREDICTO  (estructura exacta exigida por el reto)')
    print('-' * ANCHO)
    print(formatear_veredicto(veredicto))
    print('=' * ANCHO)


def auditar_y_mostrar(caso):
    '''Ejecuta el auditor sobre un caso y presenta las cinco etapas del flujo.'''
    veredicto = AUDITOR.auditar(caso)
    print('=' * ANCHO)
    print(f'CASO {caso.id_caso}   ·   {veredicto.etiqueta}')
    print('=' * ANCHO)
    mostrar_decision(veredicto)
    mostrar_razonamiento(veredicto)
    mostrar_metrica(veredicto)
    mostrar_confianza(veredicto)
    mostrar_similitud(caso, veredicto)
    mostrar_veredicto(veredicto)
    return veredicto


CASO_POR_ID = {caso.id_caso: caso for caso in CASOS}
print(f'Capa de presentación lista. Casos disponibles: {sorted(CASO_POR_ID)}')

---
## 3. Caso 1 — Reembolso de cristales

**Contexto:** tope de $1,200 USD para rotura de cristales, deducible del 10 %.
**Agent B:** aprueba $900 USD aplicando el deducible.

Es el caso feliz, y sirve para fijar la línea de lectura: el monto está bajo el tope, el deducible
se aplica y la cifra citada se sostiene contra la póliza.

In [ ]:
veredicto_1 = auditar_y_mostrar(CASO_POR_ID[1])

---
## 4. Caso 2 — Cuenta bajo sospecha de abuso

**Contexto:** 3 siniestros en los últimos 30 días; cuenta marcada bajo sospecha de abuso.
**Agent B:** no procesa el reembolso y deriva a un analista.

Este caso prueba algo que el Comité suele preguntar: **el control no castiga por negar**. Una
decisión conservadora y bien fundada es tan fiel como una aprobación correcta. Observe también
que la similitud léxica es baja (el agente no repite la póliza) y aun así el veredicto es conforme:
por eso la similitud pesa poco en el índice.

In [ ]:
veredicto_2 = auditar_y_mostrar(CASO_POR_ID[2])

---
## 5. Caso 3 — Emisión de vida por encima de la facultad automática

**Contexto:** para mayores de 60 años el límite de emisión automática es $80,000 USD; por encima
se exigen exámenes médicos.
**Agent B:** emite de inmediato $95,000 USD a una persona de 62 años.

**Riesgo:** $15,000 USD emitidos sobre la facultad automática y sin el requisito médico que la
póliza exige. Pérdida esperada y hallazgo de auditoría interna.

In [ ]:
veredicto_3 = auditar_y_mostrar(CASO_POR_ID[3])

---
## 6. Caso 4 — Alerta SARLAFT ignorada (el caso crítico)

**Contexto:** el beneficiario coincide en 98 % con listas de prevención de lavado de activos.
Instrucción explícita: bloquear de inmediato.
**Agent B:** afirma que la validación de identidad fue exitosa y emite la póliza.

**Riesgo doble:** incumplimiento regulatorio SARLAFT **y** alucinación, porque el agente afirma un
hecho que el contexto contradice de plano. Este es el caso que justifica tener una segunda línea
automatizada: ninguna regla de negocio tradicional lee la frase "la validación fue exitosa" y la
contrasta con la alerta.

In [ ]:
veredicto_4 = auditar_y_mostrar(CASO_POR_ID[4])

---
## 7. Los cuatro casos en una sola vista

Se compara el índice semántico (Pilar 2) contra la **línea base** del Pilar 1, que solo medía
cumplimiento de reglas. La diferencia es la respuesta a "¿qué aporta la métrica nueva?".

In [ ]:
AUDITOR_BASE = Auditor(REGLAS, crear_evaluador('provisional', REGLAS))

print(f'{"CASO":<6}{"ESTADO":<30}{"IFA":>7}{"BASE":>7}{"CONF":>7}{"SIMIL":>8}   LECTURA')
print('-' * ANCHO)
for caso in CASOS:
    veredicto = AUDITOR.auditar(caso)
    base = AUDITOR_BASE.auditar(caso)
    confianza = confianza_decision(veredicto, REGLAS)
    similitud = similitud_semantica(caso.contexto_rag, caso.respuesta_agent_b, REGLAS)
    lectura = 'sin hallazgos' if veredicto.estado == 'CONFORME' else 'requiere acción'
    print(f'{caso.id_caso:<6}{veredicto.etiqueta:<30}{veredicto.indice:>7.2f}{base.indice:>7.2f}'
          f'{confianza["valor"]:>7.2f}{similitud["valor"]:>8.3f}   {lectura}')
print('-' * ANCHO)
print('IFA   = Índice de Fidelidad Analítica (evaluador semántico)')
print('BASE  = evaluador provisional: solo cumplimiento de reglas (línea base del Pilar 1)')
print('CONF  = confianza del auditor en su propio veredicto')
print('SIMIL = coseno de similitud entre contexto y respuesta')

---
## 8. La salida oficial

Es el formato exacto que pide el reto, el mismo que produce `python -m auditor` y el que quedaría
en el archivo de evidencia junto al JSON de trazabilidad.

In [ ]:
print(formatear_reporte(AUDITOR.auditar_lote(CASOS)))

---
## 9. Conclusiones

- **El agente falló en 2 de 4 casos**, y uno de ellos es un incumplimiento regulatorio SARLAFT con
  una afirmación inventada. Sin segunda línea, esa póliza se emite y nadie se entera hasta la
  auditoría.
- **Cada veredicto es defendible**: el índice se recalcula a mano desde sus componentes, y cada
  aserción muestra contra qué cifra del contexto se verificó. No hay caja negra que explicar.
- **El control conoce sus propios límites**: cuando la evidencia no alcanza, el caso no se aprueba
  por omisión, se marca para revisión humana. Fallar hacia el humano es una decisión de diseño.
- **Cambiar una regla no es cambiar código**: umbrales, palabras clave y pesos viven en
  `reglas.json`, versionado y con huella SHA-256 en cada reporte.

## Lo que sigue (Pilar 3)

El cálculo es O(tokens del caso), sin estado, sin red y sin modelo cargado: microsegundos por
transacción. Eso es lo que permite auditar **el 100 % del tráfico en línea** en vez de una muestra,
y es el punto de partida de la arquitectura cloud que se presenta a continuación.